# Load AI-READI clinical tables

Set the dataset path below, then run the cells in order. This notebook reads your local files. It shows clinical mapping, a participant-by-feature table and CSV export. Saved outputs are cleared in this copy.

In [ ]:
from pathlib import Path
import pandas as pd
from aireadi_preprocessing import load_clinical
from aireadi_preprocessing.tables import read_table, measurement_mapping

dataset = Path("/path/to/ai-readi")
output = Path("/path/to/local-tables")

## Inspect the clinical codes

`measurement.csv` carries the measurement code, source label and unit on each row. The [AI-READI OMOP mapping table](https://docs.aireadi.org/docs/3/dataset/clinical-data/OMOP-Clinical-Data-Structure/OMOP-Mapping-Table) explains the codes; its [source dictionary](https://github.com/AI-READI/ai-readi-docs/blob/main/docs/static/json/mappings.json) contains the target concept IDs and definitions. The [lab dictionary](https://github.com/AI-READI/ai-readi-docs/blob/main/docs/static/json/clinicalLabData.json) includes lab names and units.

The function below extracts the numeric code/label/unit pairs from your measurement file. It stops if a code has conflicting labels or units.

- Documentation `TARGET_CONCEPT_ID` matches `measurement_concept_id` in the data.
- `TARGET_CONCEPT_NAME` explains the measurement; lab `Name` and `Units` provide the short label and unit.
- Default output labels come from `measurement_source_value` in your file. Values come from `value_as_number`, indexed by `person_id`.


In [ ]:
measurements = read_table(dataset / "clinical_data" / "measurement.csv")
mapping = measurement_mapping(measurements)
mapping.head()

## Load the clinical table

The package matches measurement rows by code and pivots their numeric values into columns. Participant IDs remain strings, so leading zeros are preserved. A repeated participant/code pair stops the pivot; choose a visit or an aggregation before continuing.

In [ ]:
clinical = load_clinical(dataset)
clinical.head()

## Choose measurements and column names

This selection uses HbA1c, code `3004410`, with unit `%`. Change the rows to choose other measurements after checking the mapping. An absent measurement produces an empty column; a unit mismatch raises an error.

In [ ]:
selected_mapping = pd.DataFrame([
    {"concept_id": "3004410", "name": "HbA1c (%)", "unit": "%"},
])
selected = load_clinical(dataset, mapping=selected_mapping)
selected.head()

## Save and reload a CSV

Choose a local output directory outside the source dataset and repository. Keep `person_id` as text when reading the result back.

In [ ]:
output.mkdir(parents=True, exist_ok=True)
selected.to_csv(output / "clinical.csv", index=False)
selected_mapping.to_csv(output / "clinical_mapping.csv", index=False)
reloaded = pd.read_csv(output / "clinical.csv", dtype={"person_id": str})
reloaded.head()